[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/03_Training_Strategies/01_contrastive_learning.ipynb)

# 01. Contrastive Learning — Deep Dive

**This is the #1 training strategy for multimodal models.**

**This notebook covers:**
- InfoNCE loss — math + code + visual
- Temperature parameter — what it does and why it matters
- Hard negatives — why they improve training
- Training dynamics visualized step by step

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/03_Training_Strategies")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *

set_style()

## 1. InfoNCE Loss — The Math Made Visual

Given a batch of N image-text pairs:

$$\mathcal{L}_{\text{InfoNCE}} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(\text{sim}(I_i, T_i) / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(I_i, T_j) / \tau)}$$

Where:
- $\text{sim}(I_i, T_i)$ = cosine similarity of matching pair (positive)
- $\text{sim}(I_i, T_j)$ for $j \neq i$ = similarity with non-matching (negatives)  
- $\tau$ = temperature (controls sharpness)

In [ ]:
def infonce_loss_detailed(img_emb, txt_emb, temperature=0.07):
    """InfoNCE with step-by-step computation for visualization."""
    # Step 1: Compute cosine similarity
    img_norm = F.normalize(img_emb, dim=-1)
    txt_norm = F.normalize(txt_emb, dim=-1)
    sim_matrix = img_norm @ txt_norm.T  # [N, N]
    
    # Step 2: Scale by temperature
    logits = sim_matrix / temperature
    
    # Step 3: Labels = diagonal (matching pairs)
    N = img_emb.shape[0]
    labels = torch.arange(N)
    
    # Step 4: Cross-entropy loss (both directions)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    loss = (loss_i2t + loss_t2i) / 2
    
    # Step 5: Probabilities for visualization
    probs_i2t = F.softmax(logits, dim=1)
    
    return {
        'loss': loss,
        'sim_matrix': sim_matrix.detach(),
        'logits': logits.detach(),
        'probs': probs_i2t.detach(),
        'labels': labels
    }


# Visualize each step
N = 5
img_emb = torch.randn(N, 64)
txt_emb = torch.randn(N, 64)

result = infonce_loss_detailed(img_emb, txt_emb, temperature=0.07)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('InfoNCE Loss — Step by Step', fontsize=16, fontweight='bold')

# Step 1: Raw similarity
ax = axes[0]
sim = result['sim_matrix'].numpy()
im = ax.imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Step 1: Cosine Sim')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# Step 2: Scaled logits
ax = axes[1]
logits = result['logits'].numpy()
im = ax.imshow(logits, cmap='RdBu_r')
ax.set_title('Step 2: Logits (sim/τ)')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{logits[i,j]:.1f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# Step 3: Target
ax = axes[2]
target = np.eye(N)
ax.imshow(target, cmap='Greens')
ax.set_title('Step 3: Target Labels')
for i in range(N):
    for j in range(N):
        ax.text(j, i, '1' if i==j else '0', ha='center', va='center', fontsize=12)

# Step 4: Softmax probabilities
ax = axes[3]
probs = result['probs'].numpy()
im = ax.imshow(probs, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_title(f'Step 4: Softmax Probs\nLoss = {result["loss"].item():.3f}')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{probs[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

for ax in axes:
    ax.set_xlabel('Text')
    ax.set_ylabel('Image')

plt.tight_layout()
plt.savefig('../assets/infonce_steps.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nGoal: Make diagonal probabilities → 1.0, off-diagonal → 0.0")

## 2. Temperature — The Most Important Hyperparameter

In [ ]:
# Effect of temperature on the similarity distribution

temperatures = [0.01, 0.07, 0.5, 1.0, 5.0]
sim_scores = torch.tensor([0.8, 0.3, 0.1, -0.2, -0.5])  # similarity with 5 items

fig, axes = plt.subplots(1, len(temperatures), figsize=(20, 4))
fig.suptitle('Effect of Temperature (τ) on Probability Distribution', 
             fontsize=14, fontweight='bold')

for ax, tau in zip(axes, temperatures):
    probs = F.softmax(sim_scores / tau, dim=0).numpy()
    colors = ['#2ECC71'] + ['#E74C3C'] * 4  # positive = green
    ax.bar(range(5), probs, color=colors, alpha=0.8)
    ax.set_title(f'τ = {tau}', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(range(5))
    ax.set_xticklabels(['pos', 'neg1', 'neg2', 'neg3', 'neg4'], fontsize=8)
    
    # Annotate
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f'{p:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../assets/temperature_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print("Low τ (0.01) → Very sharp, picks the highest similarity")
print("High τ (5.0) → Very uniform, all items look similar")
print("Sweet spot (0.07) → Discriminative but not too sharp")

## 3. Training Dynamics — Watch Learning Happen

In [ ]:
# Mini experiment: train two small encoders and watch the similarity matrix evolve

class TinyEncoder(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, out_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

# Create 8 distinct pairs (each pair shares a pattern)
N = 8
torch.manual_seed(42)
img_data = torch.randn(N, 32)
txt_data = torch.randn(N, 32)
# Make matching pairs more similar in hidden structure
shared_signal = torch.randn(N, 16)
img_data[:, :16] += shared_signal * 2
txt_data[:, :16] += shared_signal * 2

img_enc = TinyEncoder(32, 16)
txt_enc = TinyEncoder(32, 16)
optimizer = torch.optim.Adam(list(img_enc.parameters()) + list(txt_enc.parameters()), lr=1e-3)

# Train and capture snapshots
snapshots = []
snapshot_epochs = [0, 5, 20, 50, 100, 200]
losses = []

for epoch in range(201):
    img_emb = img_enc(img_data)
    txt_emb = txt_enc(txt_data)
    
    logits = img_emb @ txt_emb.T / 0.07
    labels = torch.arange(N)
    loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if epoch in snapshot_epochs:
        with torch.no_grad():
            sim = (img_enc(img_data) @ txt_enc(txt_data).T).numpy()
            snapshots.append((epoch, sim))

# Plot evolution
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Contrastive Learning: Similarity Matrix Evolution', 
             fontsize=16, fontweight='bold')

for ax, (epoch, sim) in zip(axes.flat, snapshots):
    im = ax.imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'Epoch {epoch} (loss={losses[epoch]:.3f})', fontsize=12)
    ax.set_xlabel('Text')
    ax.set_ylabel('Image')
    
    diag_mean = np.mean(np.diag(sim))
    offdiag_mean = np.mean(sim[~np.eye(N, dtype=bool)])
    ax.text(0.02, 0.98, f'diag: {diag_mean:.2f}\noff: {offdiag_mean:.2f}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('../assets/training_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

# Loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color='#9B59B6', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('InfoNCE Loss')
ax.set_title('Loss Curve', fontsize=14, fontweight='bold')
for epoch in snapshot_epochs:
    ax.axvline(x=epoch, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("Watch how the diagonal gets brighter (matching pairs) and off-diagonal gets darker!")

## 4. Hard Negatives — Why They Matter

In [ ]:
# Visualize easy vs hard negatives

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Easy negatives
ax = axes[0]
ax.set_title('Easy Negatives\n(model already separates them)', fontsize=12, fontweight='bold')
# Positive pair
ax.scatter([2], [3], s=200, c='#E74C3C', marker='s', zorder=5, label='Image: cat')
ax.scatter([2.2], [3.1], s=200, c='#3498DB', marker='o', zorder=5, label='Text: "a cat"')
# Easy negatives (far away)
ax.scatter([8, 9, 7.5], [8, 7, 8.5], s=150, c='#BDC3C7', marker='x', zorder=5, label='Easy negatives')
ax.plot([2, 2.2], [3, 3.1], 'g--', linewidth=2)
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.legend()
ax.text(5, 1, 'Loss ≈ 0 (no learning signal!)', fontsize=11, color='orange', fontweight='bold')

# Hard negatives
ax = axes[1]
ax.set_title('Hard Negatives\n(similar but wrong → strong learning)', fontsize=12, fontweight='bold')
ax.scatter([5], [5], s=200, c='#E74C3C', marker='s', zorder=5, label='Image: cat')
ax.scatter([5.3], [5.2], s=200, c='#3498DB', marker='o', zorder=5, label='Text: "a cat"')
# Hard negatives (close but wrong)
ax.scatter([4.5, 5.5, 4.8], [5.5, 4.5, 4.7], s=150, c='#E67E22', marker='x', 
           zorder=5, label='Hard negatives\n("a dog", "a kitten")', linewidths=2)
ax.plot([5, 5.3], [5, 5.2], 'g--', linewidth=2)
for hx, hy in [(4.5, 5.5), (5.5, 4.5), (4.8, 4.7)]:
    ax.plot([5, hx], [5, hy], 'r:', linewidth=1, alpha=0.5)
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.legend()
ax.text(5, 1, 'Loss > 0 (strong learning signal!)', fontsize=11, color='green', fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/hard_negatives.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight: Larger batch sizes → more hard negatives → better CLIP training")
print("CLIP used batch size 32,768! But we can simulate this with techniques.")

## Key Takeaways

1. **InfoNCE loss** = softmax over similarities, matching pair as target
2. **Temperature** controls how peaked the probability distribution is (0.07 is common)
3. **Hard negatives** are critical — larger batches help find them
4. Training makes **diagonal bright, off-diagonal dark** in the similarity matrix
5. **Symmetric loss** (image→text AND text→image) works better

---
**Next:** `02_pretraining_objectives.ipynb` - Beyond contrastive learning